In [ ]:
# Import libraries

# Standard Python libraries
import os
import json
import re
from pathlib import Path
from collections import Counter

# Data handling
import pandas as pd
import numpy as np

# Progress bar
from tqdm import tqdm

# Visualisation
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Structured outputs
from pydantic import BaseModel, Field

# LLM
from langchain_groq import ChatGroq

# Kaggle dataset download
import kagglehub

#### Initializing the LLM

In [3]:
# Initialise the LLM (here, using Groq API)

from getpass import getpass

# Enter the API key when the notebook starts.

os.environ["GROQ_API_KEY"] = getpass("Groq API key: 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")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
# Reddit dataset

handle = "entenam/reddit-mental-health-dataset"

dataset_dir = Path(
    kagglehub.dataset_download(handle)
)

csv_files = sorted(
    dataset_dir.rglob("*.csv")
)

df = pd.read_csv(
    csv_files[0],
    low_memory=False
)

#### Data inspection

In [ ]:
# Dataset inspection

print(f"Number of posts: {len(df):,}")

print("\nColumns:") #score, selftext, subreddit, title, label
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nExample narrative:\n") # the first example of the self narrative
print(df.loc[0, "selftext"])

Number of posts: 223

Columns:
['score', 'selftext', 'subreddit', 'title', 'Label']

Missing values:
score        23
selftext     23
subreddit    23
title        23
Label        23
dtype: int64

Example narrative:

Tried to watch this documentary “anxious America” and it kinda just made things worse. Most of the people in the movie have houses, an education ext, ext. and they still have anxiety. How the fuck do you get better when you have no future. No funds for an education. I’m also an alcoholic cause that helps me forget about shit. But watching this and these people have anxiety, while they have an education and a license and shit. Not saying no one can have anxiety, but if someone with a life can’t get over anxiety how do I have a chance... Like I’m so much further behind these people that are so lost they made a documentary about them. Am I past the point of lost?


In [6]:
# We want to only analyse the narratives.

# Metadata (score, title, subreddit...) are intentionally excluded because the aim of the project
# is to only investigate the content of self-reported experiences.

texts = (
    df["selftext"]
    .dropna()
    .astype(str)
    .tolist()
)

print(f"\nNarratives available: {len(texts):,}")


Narratives available: 200


# Structured output schemas

In [8]:
# LLM free-text responses are difficult to analyse systematically. Every stage of the workflow here is
# thus constrained using Pydantic schemas. Each information could in this way be aggregated, visualized and reused in later stages.

### Narrative analysis

In [9]:
# Observable characteristics of each narrative without attempting to infer psychological mechanisms.

class NarrativeAnalysis(BaseModel):

    emotions: list[str]

    challenges: list[str]

    contextual_factors: list[str]

    therapeutic_needs: list[str]

### Behavioural mechanism analysis

In [11]:
# Behavioural mechanisms are inferred from the narratives. An explanation is produced.

In [10]:
class BehaviouralAnalysis(BaseModel):

    mechanisms: list[str]

    explanation: str

### Intervention mapping

In [13]:
# Based on the inferred behavioural mechanisms, the model proposes evidence-informed psychological interventions.

In [12]:
class Intervention(BaseModel):

    name: str

    rationale: str

    target_mechanisms: list[str]


class InterventionSuggestion(BaseModel):

    interventions: list[Intervention]

#### Evidence traceability

In [15]:
# Improving transparency by documenting which parts of the text motivated the generated outputs.

In [14]:
class EvidenceQuote(BaseModel):

    quote: str

    interpretation: str


class EvidenceTrace(BaseModel):

    supporting_evidence: list[EvidenceQuote]

    explanation: str

### Research hypotheses generation

In [16]:
# This generates testable hypotheses that could be used for future empirical research.

In [17]:
class Hypothesis(BaseModel):

    hypothesis: str

    rationale: str

    supporting_patterns: list[str]

    supporting_evidence: list[EvidenceQuote]

    research_value: str


class ResearchHypothesis(BaseModel):

    hypotheses: list[Hypothesis]

# Helper functions

In [18]:
def clean_json(response): # clean the first-person narratives

    response = response.strip()

    response = re.sub(
        r"^```json",
        "",
        response,
        flags=re.IGNORECASE
    )

    response = re.sub(
        r"^```",
        "",
        response
    )

    response = re.sub(
        r"```$",
        "",
        response
    )

    return response.strip()

In [19]:
def call_llm(prompt): # function for calling the LLM

    response = llm.invoke(prompt)

    return response.content

In [20]:
def run_analysis(prompt, schema): # generic analysis function

    response = call_llm(prompt)

    response = clean_json(response)

    return schema.model_validate_json(response)

#### Function for outputs saving

In [22]:
def save_json(data, filename):

    os.makedirs("outputs", exist_ok=True)

    filepath = os.path.join(
        "outputs",
        filename
    )

    with open(filepath, "w") as f:

        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False
        )

    print(f"Saved: {filepath}")

In [23]:
def save_csv(df, filename):

    os.makedirs("outputs", exist_ok=True)

    filepath = os.path.join(
        "outputs",
        filename
    )

    df.to_csv(
        filepath,
        index=False
    )

    print(f"Saved: {filepath}")

In [24]:
# output folders

folders = [

    "outputs",

    "figures"

]

for folder in folders:

    os.makedirs(
        folder,
        exist_ok=True
    )

OSError: [Errno 30] Read-only file system: 'outputs'